# S&P 500 Options: Causal DML Execution

This notebook estimates the effect of the variance-risk-premium treatment on the
return-to-expiry outcome. It declares the request through the shared causal boundary and exposes
the resolved estimand, timing, confounders, nuisance model, covariance design, and refutation
protocol before execution.

`11_model_analysis` interprets the causal estimates. This notebook validates the computation
and publishes its artifact only.

Prerequisites: `03_financial_features`, `04_model_based_features`, and `05_evaluation`.

In [1]:
"""Execute the declared S&P 500 options causal DML request."""

import polars as pl

from case_studies.sp500_options.research_workflow import open_study

In [2]:
EXECUTION_TIER = "canonical"
WORKSPACE: str = ""
PREVIEW_REDUCTIONS: dict = {}

## Declared and resolved request

A preview must declare all sample, symbol, fold, or placebo reductions. Canonical execution uses
the complete pre-holdout analysis population.

In [3]:
study = open_study(execution_tier=EXECUTION_TIER, workspace=WORKSPACE or None)
request_table = pl.DataFrame(
    {
        "method": ["dml"],
        "label": ["ret_to_expiry"],
        "config_name": ["dml"],
        "execution_tier": [EXECUTION_TIER],
    }
)
request_table

method,label,config_name,execution_tier
str,str,str,str
"""dml""","""ret_to_expiry""","""dml""","""canonical"""


In [4]:
request = study.causal(
    **request_table.row(0, named=True),
    preview_reductions=PREVIEW_REDUCTIONS,
)
resolved = request.resolve()
computation = resolved.spec["computation"]
estimand = computation["estimand"]
causal_plan = pl.DataFrame(
    {
        "treatment": [estimand["treatment"]],
        "outcome": [estimand["outcome"]],
        "confounders": [", ".join(estimand["confounders"])],
        "treatment_observed_at": [estimand["treatment_observed_at"]],
        "outcome_horizon": [estimand["outcome_horizon"]],
        "folds": [computation["cv"]["n_folds"]],
        "embargo_periods": [computation["cv"]["embargo_periods"]],
        "nuisance_model": [computation["model"]["class"]],
        "covariance": ["HAC with the outcome horizon"],
        "placebo_method": [computation["refutation"]["method"]],
        "analysis_rows": [computation["analysis_population"]["n_rows"]],
        "training_hash": [resolved.identity],
    }
)
causal_plan

treatment,outcome,confounders,treatment_observed_at,outcome_horizon,folds,embargo_periods,nuisance_model,covariance,placebo_method,analysis_rows,training_hash
str,str,str,str,str,i64,i64,str,str,str,i64,str
"""vrp_21d""","""ret_to_expiry""","""rv_21d, vrp_mom_5d, spread_pct…","""decision_timestamp""","""35 days 00:00:00""",5,35,"""sklearn.ensemble.HistGradientB…","""HAC with the outcome horizon""","""within_symbol_contiguous_block…",206651,"""4e310dbab236"""


## Execute and validate

The shared DML runner fails on missing confounders, invalid temporal folds, incomplete nuisance
fits, or a non-finite HAC standard error. A cached result must match the complete resolved
identity before it can be reused.

In [5]:
if EXECUTION_TIER == "preview" and (not WORKSPACE or not PREVIEW_REDUCTIONS):
    raise ValueError("preview execution requires WORKSPACE and PREVIEW_REDUCTIONS")
result = resolved.run()
if not result.complete or result.hash != resolved.identity:
    raise RuntimeError("causal execution did not publish the complete resolved request")

~/ml4t/public-s6-sp500_options/case_studies/utils/causal.py:1440: UserWarning: block permutation with block_size=35 cannot move 14.0% of the treatment rows: they sit in segments too short to hold two blocks, so the placebo distribution holds them at their observed values and the refutation p-value is biased toward 1. Read placebo_frozen_fraction alongside the p-value, and lower block_size or widen gap_tolerance if the frozen share is large.
  results = run_dml_analysis(


In [6]:
artifact = pl.DataFrame(
    {
        "causal_hash": [result.hash],
        "label": [resolved.spec["label"]],
        "execution_tier": [result.execution_tier],
        "complete": [result.complete],
    }
)
artifact

causal_hash,label,execution_tier,complete
str,str,str,bool
"""4e310dbab236""","""ret_to_expiry""","""canonical""",true


The registered causal artifact is the handoff to `11_model_analysis`. No estimate or empirical
conclusion is interpreted here.